# 기말 프로젝트 트랙 1 시작 노트북. VAE 의 잠재 차원
*Final project, Track 1 starter. VAE and the latent dimension*

**확인할 가설 / Hypothesis to check.** 잠재 차원이 크면 보류한 이미지의 복원 오차는 줄지만, 사전분포에서 뽑은 표본이 꼭 더 좋아 보이지는 않는다.
*A larger latent dimension lowers the reconstruction error on held-out images, but samples drawn from the prior do not necessarily look better.*

4주차 실습과 같은 VAE(784 → 512 → 256 → 잠재, 디코더는 거꾸로), MNIST, 6에폭, 시드 0 입니다. 여러분은 **잠재 차원의 값들**을 고릅니다. 자료, 망 크기, 학습 길이, 시드는 그대로 둡니다. MNIST 학습은 차원 하나에 30초~1분이라 시드 반복은 기본으로 하지 않습니다. 대신 작성칸 3 에 「무엇을 확인했고 무엇은 확인하지 못했나」 를 쓰세요. **가설이 맞지 않아도 근거를 갖춰 설명하면 같은 점수입니다.**
*The Week 4 lab's VAE (784 → 512 → 256 → latent, the decoder mirrored), MNIST, 6 epochs, seed 0. You choose **the latent dimensions**; the data, network size, training length and seed stay fixed. One MNIST training takes 30 s to a minute, so seeds are not repeated by default; say in answer 3 what your runs checked and what they did not. **A hypothesis that does not hold, explained with evidence, earns the same marks.***

**사전분포 표본을 숫자로 보는 법 / Reading prior samples with numbers.** 그림만으로 「더 좋아 보인다」 를 말하기 어려워, MNIST 로 2에폭 학습한 작은 판독기(시험 정확도 약 95%)가 표본마다 숫자를 읽습니다. 판독기가 0.9 넘게 확신한 비율과, 그 표본들에 나온 숫자 종류의 수를 셉니다. 판독기는 거친 잣대입니다. 진짜 시험 이미지에서의 같은 비율과 견주어 읽으세요.
*A small reader trained on MNIST for 2 epochs (about 95% test accuracy) reads each prior sample: the share it labels with more than 0.9 confidence, and how many digit kinds appear among those. It is a rough yardstick; compare it with the same share on real test images.*

**쓰는 법 / How to use.** 「내 선택」 칸에서 잠재 차원을 고르고(바꾸지 않으면 2 · 8 · 32) 작성칸 1 을 쓴 뒤 런타임 > 모두 실행 → 끝의 작성칸 2 ~ 4 와 AI 사용 칸에 답. 전체 2~4분(CPU).
*Choose the latent dimensions in "Your choice" (2, 8, 32 if you leave it), write answer 1, then Runtime > Run all, and fill answer cells 2 to 4 and AI use at the end. Two to four minutes on a CPU.*

In [ ]:
#@title 준비와 MNIST / Setup and MNIST { display-mode: "form" }
import math, os, gzip, struct, urllib.request, time, json, hashlib, platform, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.distributions import Normal, kl_divergence
import matplotlib.pyplot as plt
torch.set_num_threads(2)
NOTEBOOK = "FP-Track1 v6"

MNIST_FILES = ['train-images-idx3-ubyte.gz', 'train-labels-idx1-ubyte.gz', 't10k-images-idx3-ubyte.gz', 't10k-labels-idx1-ubyte.gz']
MIRRORS = ['https://ossci-datasets.s3.amazonaws.com/mnist/', 'https://storage.googleapis.com/cvdf-datasets/mnist/',
           'https://raw.githubusercontent.com/fgnt/mnist/master/']          # 4주차 실습과 같은 미러 / the Week 4 lab's mirrors
def fetch(name):
    if os.path.exists(name): return name
    errors = []
    for base in MIRRORS:
        try:
            urllib.request.urlretrieve(base + name, name); return name
        except Exception as exc:
            errors.append(f"{base}: {type(exc).__name__}")
    raise RuntimeError("MNIST 내려받기 실패 / could not download " + name + "\n  " + "\n  ".join(errors))
def idx_images(p):
    with gzip.open(p, "rb") as f:
        _, n, r, c = struct.unpack(">IIII", f.read(16)); return np.frombuffer(f.read(), np.uint8).reshape(n, r * c)
def idx_labels(p):
    with gzip.open(p, "rb") as f:
        struct.unpack(">II", f.read(8)); return np.frombuffer(f.read(), np.uint8)
for _f in MNIST_FILES: fetch(_f)
X_TRAIN = torch.from_numpy(idx_images(MNIST_FILES[0]).astype(np.float32) / 255.0)
Y_TRAIN = torch.from_numpy(idx_labels(MNIST_FILES[1]).astype(np.int64))
X_TEST = torch.from_numpy(idx_images(MNIST_FILES[2]).astype(np.float32) / 255.0)
Y_TEST = torch.from_numpy(idx_labels(MNIST_FILES[3]).astype(np.int64))
N_TRAIN = len(X_TRAIN)

class VAE(nn.Module):
    """4주차 실습의 VAE / the Week 4 lab's VAE."""
    def __init__(self, latent):
        super().__init__(); self.latent = latent
        self.enc = nn.Sequential(nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU())
        self.head_mu = nn.Linear(256, latent); self.head_logvar = nn.Linear(256, latent)
        self.dec = nn.Sequential(nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 512), nn.ReLU(), nn.Linear(512, 784))
    def encode(self, x):
        h = self.enc(x); return self.head_mu(h), self.head_logvar(h)
    def decode(self, z):
        return self.dec(z)

def train_vae(latent, epochs=6, batch=256, seed=0):
    """4주차 실습의 학습 반복문(복원항 + KL) / the lab's loop (reconstruction + KL)."""
    torch.manual_seed(seed); model = VAE(latent); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(epochs):
        perm = torch.randperm(N_TRAIN)
        for i in range(0, N_TRAIN, batch):
            xb = X_TRAIN[perm[i:i + batch]]; mu, logvar = model.encode(xb)
            q = Normal(mu, torch.exp(0.5 * logvar)); z = q.rsample()
            rec = F.binary_cross_entropy_with_logits(model.decode(z), xb, reduction="sum") / len(xb)
            kl = kl_divergence(q, Normal(torch.zeros_like(mu), torch.ones_like(mu))).sum() / len(xb)
            loss = rec + kl; opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def test_terms(model):
    """보류한 시험 이미지 10,000장에서 복원항과 KL(이미지당) / reconstruction term and KL per image on the 10,000 held-out test images."""
    g = torch.Generator().manual_seed(123); mu, logvar = model.encode(X_TEST)
    z = mu + torch.exp(0.5 * logvar) * torch.randn(mu.shape, generator=g)
    rec = F.binary_cross_entropy_with_logits(model.decode(z), X_TEST, reduction="sum").item() / len(X_TEST)
    kl = kl_divergence(Normal(mu, torch.exp(0.5 * logvar)), Normal(torch.zeros_like(mu), torch.ones_like(mu))).sum().item() / len(X_TEST)
    return rec, kl

# 판독기: MNIST 로 2에폭 학습한 작은 분류기 / the reader: a small classifier trained on MNIST for 2 epochs
torch.manual_seed(0)
reader = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)); _opt = torch.optim.Adam(reader.parameters(), lr=1e-3)
for _ in range(2):
    perm = torch.randperm(N_TRAIN)
    for i in range(0, N_TRAIN, 256):
        idx = perm[i:i + 256]; l = F.cross_entropy(reader(X_TRAIN[idx]), Y_TRAIN[idx]); _opt.zero_grad(); l.backward(); _opt.step()

@torch.no_grad()
def read(images):
    """판독기가 0.9 넘게 확신한 비율(%)과 그 이미지들의 숫자 종류 수 / % read with confidence > 0.9, and the number of digit kinds among them."""
    p = F.softmax(reader(images), 1); conf, lab = p.max(1); keep = conf > 0.9
    return 100 * keep.float().mean().item(), int((torch.bincount(lab[keep], minlength=10) > 0).sum())

with torch.no_grad(): READER_ACC = 100 * (reader(X_TEST).argmax(1) == Y_TEST).float().mean().item()
REAL_CONF, REAL_KINDS = read(X_TEST[:1000])
print(f"MNIST {tuple(X_TRAIN.shape)} / {tuple(X_TEST.shape)} | 판독기 시험 정확도 / reader test accuracy {READER_ACC:.1f}%")
print(f"진짜 시험 이미지 1,000장: 확신 비율 {REAL_CONF:.1f}%, 숫자 종류 {REAL_KINDS} / real test images: confident share {REAL_CONF:.1f}%, digit kinds {REAL_KINDS}")

## 내 선택 / Your choice
잠재 차원의 값들을 쉼표로 적으세요. 2 · 4 · 8 · 16 · 32 · 64 가운데 둘에서 넷. 기본은 2, 8, 32 입니다. 4주차 실습은 2 와 16 을 썼습니다. 둘째 시드(1)로 한 번 더 돌릴 수도 있지만 시간이 두 배가 됩니다.
*Write the latent dimensions separated by commas: two to four of 2, 4, 8, 16, 32, 64. The default is 2, 8, 32; the Week 4 lab used 2 and 16. You may also rerun with a second seed (1), which doubles the time.*

In [ ]:
#@title 내 선택 / Your choice { display-mode: "form" }
LATENTS = "2, 8, 32"  #@param {type:"string"}
ADD_SECOND_SEED = False  #@param {type:"boolean"}

ALLOWED = (2, 4, 8, 16, 32, 64)
vals, bad = [], []
for tok in [t for t in LATENTS.replace(" ", "").split(",") if t]:
    try: vals.append(int(tok))
    except ValueError: bad.append(tok)                                    # 숫자가 아닌 값 / not a number
chosen = sorted(set(vals)); bad += [v for v in chosen if v not in ALLOWED]
if bad or not 2 <= len(chosen) <= 4:
    print(f"값을 확인하세요: {LATENTS!r}. 2 · 4 · 8 · 16 · 32 · 64 중 둘에서 넷. 기본값 2, 8, 32 로 돕니다 / check the values; running the default 2, 8, 32.")
    chosen = [2, 8, 32]
SEEDS = (0, 1) if ADD_SECOND_SEED else (0,)
CHOICE = {"latents": chosen, "seeds": list(SEEDS), "epochs": 6}
print("잠재 차원 / latent dimensions:", chosen, "| 시드 / seeds:", list(SEEDS), "| 6 epochs each")

## 작성칸 1. 비교 목적과 판정 기준 / Answer 1. Purpose and decision rule
「내 선택」 을 정한 뒤, 그 설정으로 돌리기 전에 아래 칸에 쓰세요. 제안서의 평가 계획을 옮겨 와도 됩니다. 결과를 본 뒤에 고치지 말고, 결과를 본 뒤에 썼다면 그렇다고 밝히세요.
*Write this in the cell below after choosing and before you run your choice; you may carry over the evaluation plan from your proposal. Do not revise it after seeing results, and if you wrote it after seeing results, say so.*

- **목적 / Purpose.** 바꾼 조건과 값, 고정한 것, 이 비교로 답하려는 질문. 제안서와 달라진 설정이 있으면 무엇이 왜 달라졌는지.
  *The condition and values you change, what stays fixed, and the question this comparison answers; if any setting differs from your proposal, what changed and why.*
- **예측 / Prediction.** 지표마다 어느 쪽으로 움직일지와 그렇게 보는 이유.
  *For each measure, which way it will move and why you think so.*
- **판정 기준 / Decision rule.** 가설의 두 부분을 각각 무엇으로 판정할지. 특히 「사전분포 표본이 더 좋아 보인다」 를 확신 비율, 숫자 종류, 64장 그림 가운데 무엇으로 먼저 보고, 서로 어긋나면 어떻게 할지.
  *How you will judge each part of the hypothesis; in particular, which of the confident share, the digit kinds and the 64-sample grid decides "the prior samples look better", and what you do if they disagree.*

**답 / Answer.** 여기에 쓰세요. *Write here.*

In [ ]:
#@title 실행 / Run. 차원 하나에 30초~1분 / 30 s to a minute per dimension { display-mode: "form" }
rows, models, t0 = [], {}, time.time()
for seed in SEEDS:
    for lat in chosen:
        t1 = time.time(); model = train_vae(lat, seed=seed); rec, kl = test_terms(model)
        with torch.no_grad():
            prior = torch.sigmoid(model.decode(torch.randn(1000, lat, generator=torch.Generator().manual_seed(7))))
        conf, kinds = read(prior)
        rows.append({"latent": lat, "seed": seed, "test_reconstruction": rec, "test_KL": kl, "test_rec_plus_KL": rec + kl,
                     "prior_confident_%": conf, "prior_digit_kinds": kinds, "seconds": time.time() - t1})
        models[(lat, seed)] = model
        print(f"latent {lat:2d} seed {seed}: reconstruction {rec:.2f} | KL {kl:.2f} | prior confident {conf:.1f}% | kinds {kinds}")
RESULTS = pd.DataFrame(rows)
RESULTS.to_csv("track1_results.csv", index=False)
print(f"완료 / done in {time.time() - t0:.0f} s. 결과는 track1_results.csv 에도 저장 / also saved to track1_results.csv")

In [ ]:
#@title 요약 / Summary { display-mode: "form" }
print("SUMMARY. 보류 이미지와 사전분포 표본 / held-out images and prior samples")
print(RESULTS.drop(columns="seconds").round(2).to_string(index=False))
print(f"\nREAL TEST IMAGES. 판독기 확신 비율 {REAL_CONF:.1f}%, 숫자 종류 {REAL_KINDS} / reader confident share {REAL_CONF:.1f}%, digit kinds {REAL_KINDS}")
print("읽는 법: 복원항은 작을수록 보류 이미지를 잘 되살립니다. 확신 비율은 사전분포 표본이 숫자로 읽히는 정도의 거친 잣대입니다.")
print("How to read: a lower reconstruction term means held-out images are rebuilt better; the confident share is a rough yardstick for how digit-like the prior samples are.")

In [ ]:
#@title 그림 / Figures { display-mode: "form" }
seed0 = SEEDS[0]
show = X_TEST[:8]
fig, axs = plt.subplots(len(chosen) + 1, 8, figsize=(8, 1.05 * (len(chosen) + 1)))
for j in range(8):
    axs[0, j].imshow(show[j].view(28, 28), cmap="gray"); axs[0, j].axis("off")
axs[0, 0].set_title("held-out originals", fontsize=8, loc="left")
for i, lat in enumerate(chosen, 1):
    with torch.no_grad():
        mu, _ = models[(lat, seed0)].encode(show); recon = torch.sigmoid(models[(lat, seed0)].decode(mu))
    for j in range(8):
        axs[i, j].imshow(recon[j].view(28, 28), cmap="gray"); axs[i, j].axis("off")
    axs[i, 0].set_title(f"latent {lat}: reconstructions", fontsize=8, loc="left")
plt.tight_layout(); plt.show()
fig, axs = plt.subplots(1, len(chosen), figsize=(3.2 * len(chosen), 3.4))
for ax, lat in zip(np.atleast_1d(axs), chosen):
    with torch.no_grad():
        imgs = torch.sigmoid(models[(lat, seed0)].decode(torch.randn(64, lat, generator=torch.Generator().manual_seed(7)))).view(8, 8, 28, 28)
    ax.imshow(imgs.permute(0, 2, 1, 3).reshape(8 * 28, 8 * 28), cmap="gray"); ax.axis("off")
    r = RESULTS[(RESULTS["latent"] == lat) & (RESULTS["seed"] == seed0)].iloc[0]
    ax.set_title(f"latent {lat}: 64 prior samples\nconfident {r['prior_confident_%']:.0f}%, kinds {int(r['prior_digit_kinds'])}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
#@title 기록 / Record. 출력을 지우지 마세요 / keep this output { display-mode: "form" }
# 결과 해시는 같은 결과인지 대조하는 용도입니다. 실행 시간은 빼고 셉니다 / the results hash is for comparing results; run times are left out
record = {"notebook": NOTEBOOK, "choice": CHOICE,
          "results_sha256": hashlib.sha256(RESULTS.drop(columns=["seconds"], errors="ignore").to_csv(index=False).encode()).hexdigest()[:16],
          "python": platform.python_version(), "torch": torch.__version__, "device": "CPU (GPU available: " + ("yes" if torch.cuda.is_available() else "no") + ")",
          "run_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}
if "RUN_EXTENSION" in globals():                                   # 트랙 3 의 선택 확장 / Track 3's optional extension
    record["extension"] = ({"p_drop": float(EXT_P_DROP), "results_sha256": hashlib.sha256(EXT_RESULTS.to_csv(index=False).encode()).hexdigest()[:16]}
                           if RUN_EXTENSION and "EXT_RESULTS" in globals() else "not run")
print(json.dumps(record, ensure_ascii=False, indent=2))

## 이 노트북이 보고서입니다 / This notebook is your report
아래 작성칸 2 ~ 4 와 AI 사용 칸에 답을 쓰고, 출력을 저장한 채 내세요. 숫자는 위 표에서 그대로 인용합니다. 위 기록 셀의 출력은 지우지 마세요(재현성 1점).
*Answer in cells 2 to 4 and the AI-use cell below, and submit with the outputs saved. Quote numbers from the tables above. Keep the output of the record cell above (reproducibility, 1 point).*

포스터에는 요약 표와 그림 하나를 옮기면 됩니다. 표와 그림을 옮긴 것만으로는 분석 점수를 받지 않습니다.
*For the poster, copy the summary table and one figure. Copying tables and figures alone does not earn the analysis marks.*

## 작성칸 2. 내 결과와 판정 / Answer 2. Your results and judgement
- **관찰 / Observation.** 표와 그림에서 본 것을 숫자로. 기준 설정과 바꾼 설정을 나란히.
  *What the tables and figures show, in numbers, with the baseline next to your changed setting.*
- **가능한 설명 / Possible explanation.** 왜 그렇게 나왔다고 보는지. 관찰과 구분하고, 확인하지 않은 설명이면 그렇다고 씁니다.
  *Why you think it came out this way, kept apart from the observation; say so if you have not tested the explanation.*
- **판정 / Judgement.** 작성칸 1 의 판정 기준으로 가설이 맞았는지. 가장 중요한 결론을, 그것을 받치는 숫자 둘과 그 숫자가 나온 설정으로 씁니다(필수 문단).
  *Did the hypothesis hold under your decision rule in answer 1? Give your main conclusion with the two numbers that support it and the setting they come from (the required paragraph).*
- 진짜 시험 이미지의 확신 비율(REAL TEST IMAGES 줄)과 견주어 읽으세요.
  *Read the confident share against the real test images (the REAL TEST IMAGES line).*

**답 / Answer.** 여기에 쓰세요. *Write here.*

## 작성칸 3. 한계와 판단을 바꿀 증거 / Answer 3. Limits, and evidence that would change your judgement
- 이 실행이 확인한 것과 확인하지 못한 것. 예를 들어 표집 시드만 바꾼 것은 학습이 안정적인지를 확인하지 않습니다.
  *What these runs checked and what they did not; for example, changing only the sampling seed does not test whether training is stable.*
- 결론이 맞지 않을 수 있는 상황.
  *Situations where the conclusion may not hold.*
- 어떤 증거가 나오면 판단을 바꾸겠는지.
  *What evidence would make you change your judgement.*
- 노트북을 바꿨다면 무엇을 바꿨는지(예: 시드 추가).
  *Anything you changed in the notebook, such as adding seeds.*

**답 / Answer.** 여기에 쓰세요. *Write here.*

## 작성칸 4. 받은 질문과 답 / Answer 4. The question you received, and your answer
12/16 까지 포스터 댓글로 받은 질문 하나를 옮겨 적고, 이 노트북의 결과를 근거로 답하세요.
*Copy one question you received in the comments on your poster by 16 Dec, and answer it with evidence from this notebook.*

**답 / Answer.** 여기에 쓰세요. *Write here.*

## AI 사용 / AI use
도구, 용도, 대략의 범위를 쓰세요. 쓰지 않았으면 「없음」. 결론과 그 근거는 본인의 판단이어야 합니다.
*The tool, its purpose and the rough extent; write "none" if you used none. The conclusions and the evidence for them must be your own judgement.*

**답 / Answer.** 여기에 쓰세요. *Write here.*